# Geolocation Convolutional Neural Network (CNN)

The following websites were used for image data:\
https://www.kaggle.com/datasets/habedi/large-dataset-of-geotagged-images/data \
https://www.kaggle.com/code/pkompally/location-cnn

In [ ]:
# Import statements
import marimo as mo
import s2sphere as s2
import torch
import torchvision
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader
from torchvision.transforms import ToTensor
from torchvision import transforms
from io import BytesIO
import msgpack
from PIL import Image
from torch.utils.data import IterableDataset
import os
import csv
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
import hashlib
import time

In [ ]:
# Initialzes file metatdata.csv to help speed up late computation

all_the_rows = []
path = r"data"
global_idx = 0
# If data/metadata.csv exists, delete it.
if os.path.exists(os.path.join(path, "metadata.csv")):
    os.remove(os.path.join(path, "metadata.csv"))

# Loops through every image in "data" directory, collects metadata from each image.
for file in os.listdir(path):
     # Opens a .MSG file, contents is in binary so binary read is used.
     with open(os.path.join(path, file), "rb") as f:
        # Converts binary to list of dictionaries
        data = msgpack.Unpacker(f, raw=False)
        file_name = f.name
        for idx, item in enumerate(iter(data)):
            d = {
                # The global index, including all files
                "global_idx": global_idx,
                # Name of the file
                "file_path": file_name,
                # Index only within the file
                "local_idx": idx,
                # Latitude
                "latitude": item['latitude'],
                # Longitude
                "longitude": item['longitude']
            }

            global_idx += 1
            # Appends metadata to a list
            all_the_rows.append(d)

# Writes all_the_rows metatdata to the file "metadata.csv"
with open(os.path.join(path, 'metadata.csv'), 'w', newline='') as output_file:
    keys = all_the_rows[0].keys()
    dict_writer = csv.DictWriter(output_file, keys)
    dict_writer.writeheader()
    dict_writer.writerows(all_the_rows)

## Creating an S2 cell grid

In [ ]:
# Creates an s2 cell grid using recursion, with more cells in areas with more photos
def build_adaptive_grid(current_cell, cell_photos, max_level, max_photos):
    # Base cases
    # Stops if the current cell level reaches max level and returns
    if current_cell.level() >= max_level:
        return [current_cell]

    # Stops if the number of cells photos is less than the max number of photos and returns
    if len(cell_photos) <= max_photos:
        return [current_cell]

    # Recursion
    final_cells = []

    # Generates 4 child cells
    for i in range(4):
        child_cell = current_cell.child(i)

        # Creates a new "cell_photos" call child_photos. child_photos only contains cell_photos which fall into the child cell
        child_photos = [
            photo_id for photo_id in cell_photos 
            if child_cell.contains(photo_id)
        ]

        # If the child has 0 photos, ignore it
        if not child_photos:
            continue

        # Recurse into the child
        deep_cells = build_adaptive_grid(child_cell, child_photos, max_level, max_photos)
        final_cells.extend(deep_cells)

    return final_cells


def generate_global_grid(raw_coordinates, max_level=15, max_photos=100):
    print(f"# of Photos: {len(raw_coordinates)}")

    # Converts coordinates to cell_ids
    photo_cell_ids = [
        s2.CellId.from_lat_lng(s2.LatLng.from_degrees(lat, long)) for lat, long in raw_coordinates
    ]

    # Creates six base faces of Earth
    base_faces = [s2.CellId.from_face_pos_level(face, 0, 0) for face in range(6)]

    all_final_cells = []

    # Feed each face into the recursive function
    for face_cell in base_faces:

        # Filter photos for just this face
        face_photos = [p for p in photo_cell_ids if face_cell.contains(p)]

        if not face_photos:
            continue 

        # Runs the recursive builder for this face
        cells = build_adaptive_grid(face_cell, face_photos, max_level, max_photos)
        all_final_cells.extend(cells)

    print(f"Created grid of {len(all_final_cells)} cells/classes.")
    return all_final_cells

In [ ]:
# Grabs all coordinate data from metadata.csv.
coords = pd.read_csv(os.path.join(path, 'metadata.csv'))
coords = list(zip(coords['latitude'], coords['longitude']))

"""
Calls generate_global_grid
max_level: Max level the cells will recurse to. level 0 is the bottom level, with six cells making up the entrirety of Earth.
max_photos: If the number of photos in a cell drops below max_photos, recursion will stop.
"""
final_s2_cells = generate_global_grid(coords, max_level=2, max_photos=10000)

# Maps each cell ID to its corresponding class.
cell_id_dict = dict(zip(final_s2_cells, list(range(len(final_s2_cells)))))

# Number of S2 cells generated.
cell_num = len(final_s2_cells)

In [ ]:
# Retrieve class ID from cell
def get_class_id(row, s2_cells):
    # Convert lat/lng to Leaf Cell
    leaf = s2.CellId.from_lat_lng(s2.LatLng.from_degrees(row['latitude'], row['longitude']))

    # Check which parent cell contains it
    for parent_cell in s2_cells:
        if parent_cell.contains(leaf):
            # Returns class ID
            return cell_id_dict[parent_cell]
    return -1 # Fallback if something goes wrong

## Loading the data

In [ ]:
# Class that makes a GeoDataset specifically for the geolocation use case. Inherits torch.utils.data.IterableDataset
class GeoDataset(IterableDataset):
    # Initializes class
    def __init__(self, path, class_map, transform=None, split = 'train', split_ratio = 0.8, show_image = False):
        # path: file path to the file containing the metadata.csv and data folders. Initialized at the start.
        self.path = path
        # transform: A pytorch transform list which tells the 
        self.transform = transform
        
        # Dictionary mapping cells to class IDs.
        self.class_map = class_map

        # Specifies data split (train or test)
        self.split = split
        # Split ratio
        self.split_ratio = split_ratio

        # Specifies whether images should be returned as output
        self.show_image = show_image

    # Defines iterable functionality
    def __iter__(self):
        for file_name in os.listdir(self.path):
            # Only loops over .msg files
            if not file_name.endswith('.msg'):
                continue
            # Opens file in binary read mode
            with open(os.path.join(self.path, file_name), "rb") as file:
                    # Creates unpacker object, converting binary into a list of dictionaries
                    unpacker = msgpack.Unpacker(file, raw=False)
                    for item in unpacker:
                        
                        # Deterministic Splitting Logic
                        # We hash the image bytes to get a consistent value between 0 and 1
                        data_hash = hashlib.md5(item['image']).hexdigest()
                        # Convert hex to a float between 0 and 1
                        val = int(data_hash, 16) / float(1 << 128)

                        if self.split == 'train':
                            if val > self.split_ratio: continue # Skip if in test range
                        else: # test
                            if val <= self.split_ratio: continue # Skip if in train range

                        # Grabs lat/long, and gets cell ID from this info
                        lat, lng = item['latitude'], item['longitude']
                        leaf = s2.CellId.from_lat_lng(s2.LatLng.from_degrees(lat, lng))

                        # Initialize class label.
                        label_int = -1

                        # Changes label_int to the class ID of cell if lat/long if inside of a cell.
                        for level in range(15, -1, -1):
                            parent_cell = leaf.parent(level)
                            if parent_cell in self.class_map:
                                label_int = self.class_map[parent_cell]
                                break

                        if label_int == -1:
                            continue # Skip if outside our grid


                        image = Image.open(BytesIO(item['image']))

                        # To show image, comment out these last two lines.
                        if self.show_image == False:
                            if self.transform != None:
                                image = self.transform(image)

                        yield image, label_int

# Generalized Mean Pooling implementation. After attempting to train with this, I was getting subpar results so either the implementation is wrong or something in my model layers was wrong.
class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super(GeM, self).__init__()

        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        x_p = x.clamp(min=self.eps).pow(self.p)

        pooled = F.avg_pool2d(x_p, (x_p.size(-2), x_p.size(-1)))

        return pooled.pow(1.0 / self.p)

In [ ]:
# Creates a trnasform pipeline for images to help with model training.
transform = transforms.Compose([
    transforms.Resize((299, 299)), # Height, Width
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Creates a train dataset
train_dataset = GeoDataset(path=path, class_map = cell_id_dict, transform=transform, split = 'train')

# Creates a test dataset
test_dataset = GeoDataset(path=path, class_map = cell_id_dict, transform=transform, split = 'test')

# Creates a train dataloader using the train_dataset
train_dataloader = DataLoader(train_dataset, batch_size = 64)
# Creates a test dataloader using the test_dataset
test_dataloader = DataLoader(test_dataset, batch_size = 64)

In [ ]:
# The ResNet 50 model is used and initialized with pretrained weights.
weights = torchvision.models.ResNet50_Weights.IMAGENET1K_V2
model = torchvision.models.resnet50(weights = weights)

# Generalized Mean Pooling block. This is not used in training currently
# model.avgpool = nn.Sequential(
#     GeM(),
#     nn.Flatten()
# )

# Changes the last layer (fully-connected layer) to have the number of cell classes.
model.fc = nn.Linear(2048, cell_num)

# Creates a fully-connected block. This is not currently used in training.
# model.fc = nn.Sequential(
#     nn.Linear(2048, 1024),
#     nn.BatchNorm1d(1024),
#     nn.ReLU(),
#     nn.Dropout(p=0.5),
#     nn.Linear(1024, cell_num)
# )

# gets a list of the fully-connected layers/parameters
head_params = list(model.fc.parameters())

# Gets ID from fully-connected layers
head_param_ids = [id(p) for p in head_params]

# Gets a list of backbone layers/parameters
backbone_params = [p for p in model.parameters() if id(p) not in head_param_ids]

# Standard Gradient Descent optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3, momentum=0.9)

# AdamW optimizer with custom learning parameters for backbone/head layers. Not used in training currently.
# optimizer = torch.optim.AdamW([
#     #{'params': backbone_params, 'lr': 1e-5},
#     {'params': head_params, 'lr': 1e-3}
# ], weight_decay = 1e-4)

# CrossEntropyLoss criterion: Calculates loss
criterion = nn.CrossEntropyLoss()

# Switches to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")

# Switches the model to GPU
model = model.to(device)

In [ ]:
# Initialized the model in training mode.
model.train()

# Number of training loops
num_epochs = 10
epoch_plot = []
loss_plot = []

# Starts training loop
for epoch in range(num_epochs):
    running_loss = 0.0
    for inputs, labels in train_dataloader:
        # Gets inputs and labels from train_dataloader
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Zeroes out the gradient of optimizer
        optimizer.zero_grad()
        # Gets outputs/predictions from model inputs
        outputs = model(inputs)
        # Calculates loss from generated outputs
        loss = criterion(outputs, labels)
        # Tells the model which weights/biases need to be changed/stay the same
        loss.backward()
        # Updates weights/biases
        optimizer.step()

        # Adds current loss to running loss
        running_loss += loss.item()

    # Prints the data for each epoch. The hardcoded 30000 should be changed in the future.
    print(f"Epoch {epoch+1}, Loss: {running_loss/30000}, Time:{time.time}")
    epoch_plot.append(epoch+1)
    loss_plot.append(running_loss/30000)

In [ ]:
# Function to evaluate the trained model
def evaluate_model():
    # Initializes evaluation mode.
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    # We are not training, so disable gradient as it's not needed
    with torch.no_grad():
        # Grabs inputs and labels from test_dataloader
        for inputs, labels in test_dataloader:

            inputs = inputs.to(device)
            labels = labels.to(device)
            # Gets outputs from inputs
            outputs = model(inputs)

            # Calculates loss from outputs
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            # Gets the class which is most likely from each output in outputs
            _, predictions = torch.max(outputs, 1)

            # Checks which are correct and which aren't
            correct += torch.sum(predictions == labels).item()
            total += labels.size(0)

    # Something went wrong, no data found
    if total == 0:
        print("Warning: No test data found. Check your dataset or split logic.")
        return 0.0, 0.0

    test_loss = running_loss / total
    test_accuracy = correct / total
    # Prints the test loss and test accuracy.
    print(f'Test Loss: {test_loss:.4f} | Test Accuracy: {test_accuracy:.4f} ({correct}/{total})')

    return test_loss, test_accuracy

evaluate_model()

In [ ]:
# The the model loss over epochs
plt.plot(epoch_plot, loss_plot)
plt.show()

In [ ]:
# Checks what a single image looks like using show_image = True
example_dataloader = GeoDataset(path=path, class_map = cell_id_dict, transform=transform, split = 'test', show_image = True)

def show_image(image_idx):
    for idx, i in enumerate(iter(example_dataloader)):
        if idx == image_idx:
            return i

